# Croissance optimale — Notebook de cours

**Objectifs**
1. Charger le module local `OptimalGrowth.py`.
2. Simuler une économie de référence.
3. Calculer et afficher la fenêtre d’optimisation effective du planificateur.
4. Résoudre des exercices dans les cellules prévues.
5. Étudier la sensibilité aux paramètres et à la longueur de la fenêtre d’optimisation.

> Ce notebook est autonome et utilise le fichier local `OptimalGrowth.py` fourni avec le cours.


## 1) Préparation et importations


In [ ]:
# Exécutez cette cellule une fois pour vérifier/installer les paquets Python requis pour ce portable.

import sys
import subprocess
import importlib.util

required = {
    "matplotlib": "matplotlib",
    "numba": "numba",
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "tqdm": "tqdm",
}

missing = [
    package
    for module, package in required.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", *missing]
    )

print("Python environment ready.")


In [ ]:
# Si vous exécutez ce carnet ailleurs, adaptez le chemin afin qu'il puisse trouver OptimalGrowth.py
import sys, os, importlib, math
from pathlib import Path
import OptimalGrowth  as og


## 2) Paramètres du modèle (`Params`)


In [ ]:
# Inspecter les paramètres par défaut
p = og.Params()
print(p)
print("\nNombre de périodes nT =", p.nT, "avec un pas Δ =", p.Delta, "années")
print("Time runs from", p.t0, "to", p.tT)

## 3) Initialiser l’état et construire la trajectoire exogène

Avant de résoudre le problème du planificateur, il faut définir l’environnement économique :

1. les **variables exogènes**, comme la productivité et la population, dont les trajectoires sont fixées à l’avance ;
2. les **variables d’état initiales**, comme le stock de capital, qui déterminent le point de départ de l’économie.

### Définir les variables exogènes

Le vecteur exogène est $x_t \in \mathbb{R}^{N_x}$ et contient ici la productivité $A_t$ et la population $L_t$. Il suit le processus

$$
x_t = g(x_{t-1}), \quad g: \mathbb{R}^{N_x} \to \mathbb{R}^{N_x}.
$$

Dans cette version simplifiée, nous fixons $A_t=L_t=1$ afin d’isoler les mécanismes essentiels. Ces trajectoires seront stockées dans la matrice $y$ ; les variables encore inconnues restent à `NaN` jusqu’à la résolution.

### Définir les variables d’état initiales

Nous initialisons ensuite les variables d’état.


In [ ]:
# Initialiser la matrice de simulation avec les moteurs exogènes et les états initiaux
sim = og.init_states(p)

# Convertir en un cadre de données bien rangé pour l'inspection (facultatif)
df0 = og.mat_to_df(sim, p)
df0.head()

## 4) Problème d’optimisation du planificateur

Le planificateur choisit une suite de taux d’épargne $s_t \in [0,1]$ afin de maximiser l’utilité intertemporelle de la consommation, sous les contraintes dynamiques de l’économie.

---

#### Fonction objectif

Le planificateur maximise l’utilité actualisée :

$$
\max_{\{s_t\}_{t=0}^{T}} \; \sum_{i=0}^{I^{*}} \frac{1}{(1+\rho)^{i }} \cdot L_{t+i} \cdot U(c_{t+i}),
$$

où :
- $\rho$ est le taux de préférence pure pour le présent ;
- $\Delta$ est le pas de temps, exprimé en années ;
- $I^{*}$ est l’horizon effectif d’optimisation, déterminé par la tolérance `toly` ;
- $U$ est une fonction d’utilité instantanée de type CRRA :

$$
U(C_t/L_t) = \frac{(c_t)^{1-\gamma}}{1-\gamma},
$$

avec $\gamma$ le coefficient d’aversion relative au risque.

---

#### Contraintes

Les choix du planificateur respectent :

- la **contrainte de ressources**
  $$Y_t = C_t + I_t,$$
  où $Y_t$ est la production, $C_t$ la consommation et $I_t$ l’investissement ;

- la **loi d’accumulation du capital**
  $$K_{t+1} = (1 - \delta) K_t + I_t,$$
  où $\delta$ est le taux de dépréciation ;

- la **fonction de production**
  $$Y_t = A_t K_t^{\alpha} L_t^{1-\alpha},$$
  où $A_t$ est la productivité, $\alpha$ la part du capital et $L_t$ le travail.

---

#### Fenêtre effective d’optimisation

L’horizon infini est approché par une fenêtre finie $T_{\text{planner}}$. Elle est choisie de façon que le poids d’actualisation devienne inférieur à `toly` :

$$
\text{tant que } \left(\frac{1}{1+\rho}\right)^{t} > \text{toly}, \quad t \mapsto T_{\text{planner}}.
$$

L’optimisation porte donc sur un horizon tronqué suffisamment long pour rendre négligeables les termes ultérieurs.

**Interprétation.** Le planificateur choisit la trajectoire du taux d’épargne $\{s_t\}$ pour arbitrer entre consommation présente et accumulation future de capital, compte tenu de l’actualisation.


In [ ]:
import numpy as np

# Initialiser les états et simuler avec le planificateur
sim0 = og.init_states(p)
timevec = np.arange(1, p.nT, dtype=int)  # optimization/evolution index

# Bounds et variable(s) de contrôle: ici taux d'épargne s t dans [0,1]
bounds = (0.0, 1.0)
control_id = [p.i_s]

sim_opt = og.run_optimal_policy(sim.copy(), timevec, p, bounds, control_id)

# Définir la taille (efficace) de la fenêtre du planificateur dérivée de p.rho, p.Delta et p.toly ( logique module).
def planner_window_size(rho, Delta, toly):
    # Reproduire la logique de boucle de og.run_optimal_policy :
    # disque commence à 1 et est multiplié par (1/(1+rho))**Delta jusqu'à ce qu'il tombe sous 'toly'.
    if rho <= -1:
        raise ValueError("rho must be > -1")
    disc = 1.0
    Tplanner = 1
    while disc > toly:
        Tplanner += 1
        disc *= (1.0 / (1.0 + rho)) ** Delta
    return Tplanner

T_planner = planner_window_size(p.rho, p.Delta, p.toly)
print(f"Effective planner window (in periods): {T_planner}")
print(f"Pas de temps Δ = {p.Delta} → fenêtre ≈ {T_planner * p.Delta} années")

### Trajectoires optimales


In [ ]:
# Règle des graphiques pour ce cahier de cours (matplotlib seulement; un graphique par figure)
import matplotlib.pyplot as plt
import numpy as np

années = sim_opt[:, p.i_time]

# s_t
plt.figure()
plt.plot(années, sim_opt[:, p.i_s], linewidth=2)
plt.title("Optimal saving rate $s_t$")
plt.xlabel("Année"); plt.ylabel("Share"); plt.grid(True)
plt.show()

# K_t
plt.figure()
plt.plot(années, sim_opt[:, p.i_K], linewidth=2)
plt.title("Capital $K_t$")
plt.xlabel("Année"); plt.ylabel("Level"); plt.grid(True)
plt.show()

# Y, C, I (trois lignes sur une figure sont acceptables; encore une figure sur une parcelle)
plt.figure()
plt.plot(années, sim_opt[:, p.i_Y], linewidth=2, label="Y")
plt.plot(années, sim_opt[:, p.i_C], linewidth=2, label="C")
plt.plot(années, sim_opt[:, p.i_I], linewidth=2, label="I")
plt.title("Output split: $Y_t = C_t + I_t$")
plt.xlabel("Année"); plt.grid(True); plt.legend()
plt.show()

# c_t per capita
plt.figure()
c = sim_opt[:, p.i_C] / np.maximum(sim_opt[:, p.i_L], 1e-12)
plt.plot(années, c, linewidth=2)
plt.title("Per‑capita consumption $c_t$")
plt.xlabel("Année"); plt.grid(True)
plt.show()

## 5) Exercices

### Exercice 1 — Reproduire la référence et vérifier la faisabilité
- Recréez la trajectoire de référence avec vos propres appels de fonctions.
- Vérifiez à chaque période la contrainte de ressources : $Y_t=C_t+I_t$.


In [ ]:
# TODO: Your code here.
# Indications :
# p1 = og.Params(); ensure p1.lg is defined; sim1 = og.init_states(p1); ...
pass

### Exercice 2 — Augmenter la dépréciation `delta`
- Augmentez `delta` de 5 points de pourcentage, par exemple de 0,06 à 0,11.
- Simulez à nouveau le modèle et comparez les trajectoires de $s_t$, $K_t$ et $c_t$.
- Donnez une brève interprétation économique.


In [ ]:
# TODO: Your code here.
# Example skeleton:
# p_hi = og.Params(); p_hi.delta = 0.11
# sim_hi = og.run_optimal_policy(og.init_states(p_hi), np.arange(1, p_hi.nT, dtype=int), p_hi, (0,1), [p_hi.i_s])
# Comparaisons des parcelles par rapport à la base de référence.
pass

### Exercice 3 — Augmenter la part du capital `alpha`
- Faites passer `alpha` de 0,30 à 0,45.
- Expliquez l’effet sur l’épargne optimale et l’accumulation du capital.


In [ ]:
# TODO: Your code here.
pass

### Exercice 4 — Ajouter une dynamique démographique de type DICE

Jusqu’ici, la population était constante : $L_t=1$. Dans les modèles intégrés d’évaluation, elle converge vers un niveau de long terme :

$$
L_{t+1} = L_t^{1-\zeta_L} \cdot L_{\infty}^{\zeta_L}.
$$

- $L_{\infty}$ est la population de long terme ;
- $\zeta_L$ détermine la vitesse de convergence vers $L_{\infty}$.

**Questions**
- Le paramètre $\zeta_L$ vaut actuellement 0. Fixez-le à 0,02.
- Comparez les résultats avec ceux obtenus pour une population constante.


In [ ]:
# TODO: Your code here.
# Vous pouvez copier et adapter og.init states dans une nouvelle fonction, puis l'utiliser au lieu de la valeur par défaut.
#p.L0   = 1.0           # initial population (scale arbitrary)
#p.Linf = 10500         # Asymptotic population (millions)
# p.lg = 0 # Paramètres du taux de croissance de la population
pass

#### Interpréter vos résultats : une croissance démographique augmente-t-elle l'épargne ? Pourquoi ?

> Vous avez écrit la réponse ici.

### Exercice 5 — Ajouter une dynamique de productivité de type DICE

Jusqu’ici, la productivité est constante : $A_t \equiv 1$ car `gA = 0`. Introduisons une croissance de la productivité qui ralentit progressivement, comme dans DICE.

**Loi d’évolution** mise en œuvre dans `OptimalGrowth.py` :

$$
A_t = A_{t-1} G_t,
\qquad\text{avec}\qquad
G_t = \frac{1}{1-g_A\exp\!\big(-\delta_A\Delta(t-1)\big)}.
$$

Le taux de croissance instantané est approximativement

$$
g_{A,t} \approx g_A\exp\!\big(-\delta_A\Delta(t-1)\big).
$$

La croissance initiale est donc proche de $g_A$, puis converge vers zéro au rythme $\delta_A$.

**Paramètres**
- `A0` : niveau initial de productivité, normalisé à 1 ;
- `gA` : taux de croissance initial, nul dans la référence ;
- `deltaA` : vitesse annuelle de ralentissement de la croissance ;
- `Delta` : nombre d’années par période.

**Étalonnage suggéré**
- $A_0=1$ ;
- $g_A=0{,}015$ ;
- $\delta_A=0{,}005$, soit un ralentissement annuel de 0,5 %.

**Travail demandé**
1. Définissez ces paramètres ; la loi d’évolution est déjà codée dans le module.
2. Résolvez à nouveau le problème et tracez $A_t$.
3. Comparez le taux d’épargne $s_t$, la production $Y_t$ et la consommation par habitant $c_t=C_t/L_t$ avec la référence à productivité constante.


In [ ]:
# 1) Définir les paramètres de la croissance du TFP qui diminue au fil du temps
p_tfp = og.Params()
p_tfp.A0     = 1.0
p_tfp.gA     = 0.015     # ≈ 1.5%/year initial TFP growth
p_tfp.deltaA = 0.005     # growth decay rate per year (toward 0)

#....
pass

### Interprétation

Comment la croissance transitoire de la productivité modifie-t-elle l’épargne optimale et le bien-être par rapport à la référence à productivité constante ?


> Vous avez écrit la réponse ici.